# 11.5 — Biodiversity and cross-analysis robustness

This final notebook reuses compatible artifacts from Notebooks 11.1–11.4, trains only missing biodiversity-assumption cases, and writes qualified descriptive screening diagnostics. These diagnostics are not statistical confirmation. Incomplete comparison groups are reported and excluded. The `test` profile uses synthetic cells; `screen` and `full` use the preserved historical feature table.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

from estonia_landuse.sensitivity.config import DEFAULT_SEEDS
from estonia_landuse.sensitivity.historical_model import SCENARIO_LABELS
from estonia_landuse.sensitivity.robustness import (
    build_robustness_report, current_artifact_identity,
    full_manifest_for_partial_resume, inventory_artifacts,
    missing_manifest_rows,
)
from estonia_landuse.sensitivity.runner import run_manifest
from estonia_landuse.sensitivity.sampling import (
    build_biodiversity_manifest, manifest_run_count, manifest_summary,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
HISTORICAL_ROOT = (
    PROJECT_ROOT.parent.parent
    if PROJECT_ROOT.parent.name == ".worktrees"
    else PROJECT_ROOT
)
PROFILE = os.environ.get("SENSITIVITY_PROFILE", "test")
N_WORKERS = int(os.environ.get("SENSITIVITY_N_WORKERS", "2"))
OVERWRITE = os.environ.get("SENSITIVITY_OVERWRITE", "false").lower() == "true"
OUTPUT_ROOT = Path(os.environ.get("SENSITIVITY_OUTPUT_ROOT", PROJECT_ROOT / "data/processed/legacy_sensitivity")).resolve()
FEATURES_PATH = Path(os.environ.get("SENSITIVITY_FEATURES_PATH", HISTORICAL_ROOT / "data/processed/learned_carbon/features_with_forest.parquet")).resolve()
REPORT_DIR = Path(os.environ.get("SENSITIVITY_REPORT_DIR", OUTPUT_ROOT / "reports" / f"robustness_{PROFILE}")).resolve()
SEEDS = (0, 1) if PROFILE == "test" else DEFAULT_SEEDS[PROFILE]
SCENARIOS = ("balanced",) if PROFILE == "test" else tuple(SCENARIO_LABELS)


In [ ]:
if PROFILE == "test":
    position = np.linspace(0.0, 1.0, 12)
    context = pd.DataFrame({
        "cell_id": np.arange(1, 13), "forest_pct": 0.35 + 0.03 * position,
        "wetland_pct": 0.10 + 0.02 * position, "agriculture_pct": 0.30 - 0.03 * position,
        "grassland_pct": 0.15 - 0.02 * position, "urban_pct": np.full(12, 0.05),
        "water_pct": np.full(12, 0.05), "protected_overlap_pct": 0.05 * position,
        "wetland_suitability": 0.2 + 0.6 * position, "opportunity_cost_proxy": 0.1 + 0.5 * position,
        "predicted_tco2_ha_yr": 2.5 + 2.0 * position, "peat_overlap_pct": 0.4 * position,
    })
    feature_columns = ["wetland_suitability", "opportunity_cost_proxy"]
else:
    if not FEATURES_PATH.exists():
        raise FileNotFoundError(f"Missing historical feature input: {FEATURES_PATH}")
    context = pd.read_parquet(FEATURES_PATH)
    feature_columns = [name for name in ("urban_pct", "agriculture_pct", "grassland_pct", "forest_pct", "wetland_pct", "water_pct", "naturalness_score", "carbon_score", "protected_overlap_pct", "wetland_suitability", "biodiversity_proxy", "opportunity_cost_proxy", "rohemeeter_norm") if name in context]
    if not feature_columns:
        raise ValueError("No preserved Notebook 10 feature columns found")


In [ ]:
expected_identity = current_artifact_identity(context, feature_columns, PROFILE)
inventory = inventory_artifacts(OUTPUT_ROOT, PROFILE, expected_identity=expected_identity)
manifest = build_biodiversity_manifest(profile=PROFILE, scenarios=SCENARIOS, seeds=SEEDS)
planned_total_runs = manifest_run_count(manifest)
manifest_summary(manifest)
display(manifest.head(12))
missing_manifest = missing_manifest_rows(manifest, inventory)
planned_new_runs = manifest_run_count(missing_manifest) if not missing_manifest.empty else 0
execution_manifest = full_manifest_for_partial_resume(manifest, missing_manifest)
print(f"Planned new biodiversity runs: {planned_new_runs}")
display(missing_manifest.head(12))


In [ ]:
if planned_new_runs:
    statuses = run_manifest(
        context, feature_columns, execution_manifest, OUTPUT_ROOT, PROFILE,
        overwrite=OVERWRITE, n_workers=min(N_WORKERS, planned_new_runs),
        progress=lambda completed, total, status: print(f"[{completed}/{total}] {status}"),
    )
    if statuses["status"].eq("failed").any():
        raise RuntimeError(statuses.loc[statuses["status"].eq("failed"), ["sample_id", "scenario", "seed", "error_message"]].to_string(index=False))
    display(statuses["status"].value_counts())
    print(f"Optimizer training executions: {statuses['status'].eq('completed').sum()}")
else:
    statuses = pd.DataFrame()
    print("All matching biodiversity artifacts already exist; nothing scheduled.")


In [ ]:
report_paths = build_robustness_report(
    OUTPUT_ROOT, REPORT_DIR, PROFILE, expected_identity=expected_identity,
)
display(pd.read_csv(report_paths["completeness"]))
display(pd.read_csv(report_paths["comparison_groups"]))
display(pd.read_csv(report_paths["rank_stability"]))
display(pd.read_csv(report_paths["parameter_importance"]))
display(pd.read_csv(report_paths["interactions"]))
display(pd.read_csv(report_paths["spatial_robustness"]).head())
conclusions = json.loads(report_paths["conclusions"].read_text(encoding="utf-8"))
display(pd.Series(conclusions, name="conclusion"))
print("Model and interaction conclusions are descriptive screening diagnostics, not statistical confirmation.")
print(f"Robustness report written to {REPORT_DIR}")
